> **Chapter 11, Part 1** | Advanced lens. **Focus:** self-similarity, scale, recursion, and why the Mandelbrot boundary is a serious teaching object rather than a pretty image.


# Fractals and the Mandelbrot Set

Fractals are useful here because they force three ideas into one frame: recursion, scale, and instability at the boundary. The later governance argument depends on those ideas, not on aesthetic fascination with the picture.

## Outputs

- a Mandelbrot renderer in pure NumPy
- a box-counting estimate on an approximate boundary mask
- a working intuition for why scale-sensitive descriptors matter

## Supporting reading

- Britannica on the Mandelbrot set: https://www.britannica.com/science/Mandelbrot-set
- Lopes and Betrouni review: https://pubmed.ncbi.nlm.nih.gov/19535282/
- NumPy reference: https://numpy.org/doc/stable/
- Matplotlib reference: https://matplotlib.org/stable/

## Failure note

If you only produce a beautiful image and cannot explain what the iteration is doing, the notebook has failed.

## How I would debug this

Start with one point in the complex plane and iterate it by hand. The visualization makes more sense once you know what counts as escape and what stays bounded.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def mandelbrot(xmin=-2.5, xmax=1.0, ymin=-1.5, ymax=1.5, width=900, height=650, max_iter=250):
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    c = x[None, :] + 1j * y[:, None]
    z = np.zeros_like(c)
    escape = np.full(c.shape, max_iter, dtype=int)
    mask = np.ones(c.shape, dtype=bool)

    for i in range(max_iter):
        z[mask] = z[mask] * z[mask] + c[mask]
        escaped = np.abs(z) > 2
        newly_escaped = escaped & mask
        escape[newly_escaped] = i
        mask &= ~escaped

    return escape


escape = mandelbrot(max_iter=300)

plt.figure(figsize=(10, 7))
plt.imshow(escape, extent=[-2.5, 1.0, -1.5, 1.5], origin="lower", cmap="magma")
plt.xlabel("Re(c)")
plt.ylabel("Im(c)")
plt.title("Mandelbrot escape-time image")
plt.colorbar(label="escape iteration")
plt.show()


In [ ]:
def approximate_boundary(escape, max_iter):
    inside = escape == max_iter
    neighbors = (
        np.roll(inside, 1, axis=0)
        & np.roll(inside, -1, axis=0)
        & np.roll(inside, 1, axis=1)
        & np.roll(inside, -1, axis=1)
    )
    return inside & ~neighbors


def box_count(binary_mask, box_size):
    rows, cols = binary_mask.shape
    count = 0
    for r in range(0, rows, box_size):
        for c in range(0, cols, box_size):
            block = binary_mask[r:r + box_size, c:c + box_size]
            if np.any(block):
                count += 1
    return count


def estimate_box_dimension(binary_mask, box_sizes):
    counts = np.array([box_count(binary_mask, s) for s in box_sizes], dtype=float)
    valid = counts > 0
    x = np.log(1 / np.array(box_sizes)[valid])
    y = np.log(counts[valid])
    slope, intercept = np.polyfit(x, y, 1)
    return slope, intercept, x, y


boundary = approximate_boundary(escape, max_iter=300)
box_sizes = [2, 4, 8, 16, 32, 64]
dimension, intercept, x, y = estimate_box_dimension(boundary, box_sizes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(boundary, cmap="gray", origin="lower")
axes[0].set_title("Approximate boundary mask")
axes[0].set_axis_off()

axes[1].scatter(x, y, color="#1f3b2f")
axes[1].plot(x, dimension * x + intercept, color="#d17a00")
axes[1].set_title(f"Box-counting estimate: slope ≈ {dimension:.3f}")
axes[1].set_xlabel("log(1 / box size)")
axes[1].set_ylabel("log(number of occupied boxes)")
plt.tight_layout()
plt.show()


## Interpretation

The object is not valuable because the number we estimated is perfect. It is valuable because the exercise teaches what a scale-sensitive descriptor is trying to measure.

That becomes important later. In enterprise settings, the question is rarely whether a dataset is literally fractal. The better question is whether irregularity, hierarchy, clustering, or defect concentration changes in a systematic way as the observation scale changes.

## Exercise

1. zoom into a smaller region by narrowing `xmin`, `xmax`, `ymin`, and `ymax`
2. increase `max_iter`
3. compare how stable the box-counting estimate looks as you change the scale range
